<a href="https://colab.research.google.com/github/prasn14/Financial-multi-factor-models/blob/main/heston_hw_pinn_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Heston-Hull-White 3-Factor PDE — Fourier Feature PINN

**Model:** European call option under stochastic volatility + stochastic rates

| Factor | Process | Parameters |
|--------|---------|------------|
| S | Geometric Brownian Motion | ρ_Sv, ρ_Sr |
| v | CIR (Heston) | κ, θ, ξ |
| r | Ornstein-Uhlenbeck (Hull-White) | a, b, σ_r |

**PDE (τ = T−t forward time):**
$$\frac{\partial V}{\partial \tau} = \frac{1}{2}vS^2 V_{SS} + \rho_{Sv}\xi v S\, V_{Sv} + \frac{1}{2}\xi^2 v\, V_{vv} + \frac{1}{2}\sigma_r^2 V_{rr} + \rho_{Sr}\sigma_r\sqrt{v}\,S\,V_{Sr} + rS\,V_S + \kappa(\theta-v)V_v + a(b-r)V_r - rV$$

**Terminal condition:** $V(\tau{=}0,\,S,v,r) = \max(S-K,\,0)$

In [1]:
# Cell 1 — Check GPU & install (torch already in Colab, nothing extra needed)
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__}  |  device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('TIP: Runtime → Change runtime type → T4 GPU for ~10x speedup')

PyTorch 2.10.0+cu128  |  device: cuda
GPU: Tesla T4


In [2]:
# Cell 2 — Imports
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import time
from IPython.display import clear_output

torch.manual_seed(42)
np.random.seed(42)
print('Imports OK')

Imports OK


In [3]:
# Cell 3 — Model parameters (edit these to experiment)
params = {
    # Heston stochastic variance
    'kappa': 2.0,    # mean-reversion speed of v
    'theta': 0.04,   # long-run variance (vol = 20%)
    'xi':    0.30,   # vol-of-vol
    'rho_sv': -0.70, # S-v correlation (typically negative)
    # Hull-White short rate
    'a':     0.50,   # mean-reversion speed of r
    'b':     0.03,   # long-run rate (3%)
    'sig_r': 0.01,   # short rate vol
    'rho_sr': -0.10, # S-r correlation
    # Option
    'K':     1.0,    # strike (S is normalised so S0 ~ 1)
    'T':     1.0,    # maturity in years
}

# Feller condition check: 2*kappa*theta > xi^2 ensures v stays positive
feller = 2 * params['kappa'] * params['theta']
xi2    = params['xi']**2
print(f"Feller condition: 2κθ = {feller:.3f} vs ξ² = {xi2:.3f}  →  {'✓ satisfied' if feller > xi2 else '✗ VIOLATED — increase kappa or theta'}")

Feller condition: 2κθ = 0.160 vs ξ² = 0.090  →  ✓ satisfied


In [4]:
# Cell 4 — Fourier Feature Embedding
class FourierFeatureMap(nn.Module):
    """
    Maps input x (N, d) → (N, 2m) via [sin(2π B x), cos(2π B x)]
    B ~ N(0, sigma^2) is sampled once and frozen (not trained).
    sigma controls frequency bandwidth — tune to PDE scale.
    """
    def __init__(self, in_dim=4, m=128, sigma=1.0):
        super().__init__()
        B = torch.randn(m, in_dim) * sigma
        self.register_buffer('B', B)  # frozen, not a parameter

    def forward(self, x):
        proj = 2 * np.pi * x @ self.B.T   # (N, m)
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=-1)  # (N, 2m)


# Cell 4b — Residual Block
class ResBlock(nn.Module):
    """Skip connection stabilises gradients through deep PDE-trained nets."""
    def __init__(self, width):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(width, width),
            nn.Tanh(),
            nn.Linear(width, width),
        )
        self.act = nn.Tanh()

    def forward(self, x):
        return self.act(x + self.net(x))


# Cell 4c — Full PINN
class HestonHWPINN(nn.Module):
    """
    Fourier Feature PINN for the Heston-Hull-White 3-factor PDE.
    Input:  (τ, S, v, r)  — forward time, stock, variance, rate
    Output: V̂(τ, S, v, r) — option price
    """
    def __init__(self, m=128, width=256, depth=4, sigma=1.0):
        super().__init__()
        self.ff    = FourierFeatureMap(in_dim=4, m=m, sigma=sigma)
        self.proj  = nn.Linear(2*m, width)
        self.blocks = nn.Sequential(*[ResBlock(width) for _ in range(depth)])
        self.out   = nn.Linear(width, 1)

    def forward(self, tau, S, v, r):
        # log(S) maps lognormal factor to ~N(0,1) scale
        x = torch.stack([tau, torch.log(S.clamp(1e-6)), v, r], dim=-1)
        h = torch.tanh(self.proj(self.ff(x)))
        h = self.blocks(h)
        return self.out(h).squeeze(-1)


model = HestonHWPINN(m=128, width=256, depth=4, sigma=1.0).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,}')
print(model)

Model parameters: 592,385
HestonHWPINN(
  (ff): FourierFeatureMap()
  (proj): Linear(in_features=256, out_features=256, bias=True)
  (blocks): Sequential(
    (0): ResBlock(
      (net): Sequential(
        (0): Linear(in_features=256, out_features=256, bias=True)
        (1): Tanh()
        (2): Linear(in_features=256, out_features=256, bias=True)
      )
      (act): Tanh()
    )
    (1): ResBlock(
      (net): Sequential(
        (0): Linear(in_features=256, out_features=256, bias=True)
        (1): Tanh()
        (2): Linear(in_features=256, out_features=256, bias=True)
      )
      (act): Tanh()
    )
    (2): ResBlock(
      (net): Sequential(
        (0): Linear(in_features=256, out_features=256, bias=True)
        (1): Tanh()
        (2): Linear(in_features=256, out_features=256, bias=True)
      )
      (act): Tanh()
    )
    (3): ResBlock(
      (net): Sequential(
        (0): Linear(in_features=256, out_features=256, bias=True)
        (1): Tanh()
        (2): Linear(in_fe

In [5]:
# Cell 5 — PDE Residual via Autograd
def pde_residual(model, tau, S, v, r, p):
    """
    Computes the Heston-Hull-White PDE residual at collocation points.
    All second-order cross-derivatives computed via torch.autograd.grad.
    Returns residual tensor (N,) — should be zero everywhere in domain.
    """
    # Require grad on all inputs
    for t in [tau, S, v, r]:
        t.requires_grad_(True)

    V = model(tau, S, v, r)

    def grad(y, x):
        return torch.autograd.grad(
            y, x,
            grad_outputs=torch.ones_like(y),
            create_graph=True,
            retain_graph=True
        )[0]

    # First-order derivatives
    V_tau = grad(V, tau)
    V_S   = grad(V, S)
    V_v   = grad(V, v)
    V_r   = grad(V, r)

    # Second-order (pure)
    V_SS  = grad(V_S, S)
    V_vv  = grad(V_v, v)
    V_rr  = grad(V_r, r)

    # Second-order (cross)
    V_Sv  = grad(V_S, v)
    V_Sr  = grad(V_S, r)

    # ── Heston-Hull-White PDE ─────────────────────────────────────────────
    # ∂V/∂τ = ½vS²V_SS + ρ_sv·ξ·v·S·V_Sv + ½ξ²·v·V_vv
    #       + ½σ_r²·V_rr + ρ_sr·σ_r·√v·S·V_Sr
    #       + r·S·V_S + κ(θ-v)·V_v + a(b-r)·V_r - r·V
    v_safe = v.clamp(min=1e-6)  # guard sqrt(v) near zero

    residual = (
        V_tau
        - 0.5  * v * S**2  * V_SS
        - p['rho_sv'] * p['xi'] * v * S * V_Sv
        - 0.5  * p['xi']**2 * v * V_vv
        - 0.5  * p['sig_r']**2 * V_rr
        - p['rho_sr'] * p['sig_r'] * torch.sqrt(v_safe) * S * V_Sr
        - r * S * V_S
        - p['kappa'] * (p['theta'] - v) * V_v
        - p['a'] * (p['b'] - r) * V_r
        + r * V
    )
    return residual


# Quick sanity check — residual shape should be (32,)
tau_test = torch.rand(32).to(device)
S_test   = torch.exp(torch.randn(32) * 0.3).to(device)
v_test   = torch.rand(32) * 0.15 + 0.01
v_test   = v_test.to(device)
r_test   = (torch.randn(32) * 0.01 + params['b']).to(device)

res_test = pde_residual(model, tau_test, S_test, v_test, r_test, params)
print(f'Residual shape: {res_test.shape}  |  untrained L2: {res_test.pow(2).mean().item():.4f}')

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:330.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Residual shape: torch.Size([32])  |  untrained L2: 1.0870


In [6]:
# Cell 6 — Collocation Sampler
def sample_interior(N, p, device):
    """Latin-Hypercube-style sampling over the PDE domain."""
    tau = torch.rand(N, device=device) * p['T']                     # τ ∈ [0, T]
    S   = torch.exp(torch.randn(N, device=device) * 0.5)            # S lognormal ~(0.6, 1.6)
    v   = torch.rand(N, device=device) * 0.20 + 0.005               # v ∈ (0.005, 0.205]
    r   = torch.randn(N, device=device) * 0.02 + p['b']             # r ~ N(b, 0.02)
    return tau, S, v, r

def sample_terminal(N, p, device):
    """τ=0 boundary: V = max(S-K, 0) payoff."""
    tau = torch.zeros(N, device=device)
    S   = torch.exp(torch.randn(N, device=device) * 0.5)
    v   = torch.rand(N, device=device) * 0.20 + 0.005
    r   = torch.randn(N, device=device) * 0.02 + p['b']
    payoff = torch.clamp(S - p['K'], min=0.0)
    return tau, S, v, r, payoff

print('Sampler OK')

Sampler OK


In [ ]:
# Cell 7 — Training (≈ 5 min on T4, 20 min on CPU)
# Reduce n_epochs to 5000 for a quick smoke-test

N_EPOCHS  = 15000    # ← lower to 3000 for a fast test
N_INT     = 4096     # interior collocation points per batch
N_BC      = 512      # terminal boundary points per batch
BC_WEIGHT = 100.0    # λ: penalty weight for BC loss
LR        = 1e-3

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

history = {'total': [], 'pde': [], 'bc': [], 'epoch': []}
t0 = time.time()

for epoch in range(1, N_EPOCHS + 1):

    # ── Interior PDE loss ────────────────────────────────────────────
    tau, S, v, r = sample_interior(N_INT, params, device)
    res = pde_residual(model, tau, S, v, r, params)
    loss_pde = res.pow(2).mean()

    # ── Terminal BC loss (τ=0) ───────────────────────────────────────
    tau0, S_bc, v_bc, r_bc, payoff = sample_terminal(N_BC, params, device)
    V_bc   = model(tau0, S_bc, v_bc, r_bc)
    loss_bc = (V_bc - payoff).pow(2).mean()

    # ── Total loss ───────────────────────────────────────────────────
    loss = loss_pde + BC_WEIGHT * loss_bc

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()

    # ── Logging ──────────────────────────────────────────────────────
    if epoch % 500 == 0:
        elapsed = time.time() - t0
        history['epoch'].append(epoch)
        history['total'].append(loss.item())
        history['pde'].append(loss_pde.item())
        history['bc'].append(loss_bc.item())
        print(f'epoch {epoch:5d}/{N_EPOCHS} | '
              f'pde {loss_pde.item():.3e} | '
              f'bc  {loss_bc.item():.3e} | '
              f'total {loss.item():.3e} | '
              f'{elapsed:.0f}s')

print(f'\nTraining complete in {time.time()-t0:.1f}s')

epoch   500/15000 | pde 1.738e-02 | bc  4.724e-04 | total 6.462e-02 | 69s
epoch  1000/15000 | pde 3.278e-03 | bc  1.886e-04 | total 2.214e-02 | 142s
epoch  1500/15000 | pde 7.678e-03 | bc  2.362e-04 | total 3.129e-02 | 214s
epoch  2000/15000 | pde 6.463e-03 | bc  8.040e-05 | total 1.450e-02 | 286s
epoch  2500/15000 | pde 5.220e-03 | bc  2.491e-04 | total 3.013e-02 | 358s
epoch  3000/15000 | pde 5.598e-03 | bc  1.234e-04 | total 1.794e-02 | 430s
epoch  3500/15000 | pde 3.166e-03 | bc  1.324e-04 | total 1.641e-02 | 502s
epoch  4000/15000 | pde 4.105e-03 | bc  1.131e-04 | total 1.542e-02 | 574s
epoch  4500/15000 | pde 4.416e-03 | bc  1.224e-04 | total 1.666e-02 | 647s
epoch  5000/15000 | pde 2.062e-03 | bc  8.010e-05 | total 1.007e-02 | 718s
epoch  5500/15000 | pde 3.113e-03 | bc  9.885e-05 | total 1.300e-02 | 790s
epoch  6000/15000 | pde 6.280e-04 | bc  3.453e-05 | total 4.081e-03 | 862s
epoch  6500/15000 | pde 1.306e-03 | bc  3.332e-05 | total 4.638e-03 | 934s
epoch  7000/15000 | pde 1.

In [ ]:
# Cell 8 — Plot training convergence
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].semilogy(history['epoch'], history['pde'],  label='PDE residual', color='#7469d4', lw=2)
axes[0].semilogy(history['epoch'], history['bc'],   label='BC loss',      color='#1D9E75', lw=2)
axes[0].semilogy(history['epoch'], history['total'],label='Total loss',   color='#D85A30', lw=2, ls='--')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss (log scale)')
axes[0].set_title('Training convergence')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history['epoch'], history['pde'],  color='#7469d4', lw=2, label='PDE')
axes[1].plot(history['epoch'], history['bc'],   color='#1D9E75', lw=2, label='BC')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss (linear)')
axes[1].set_title('Loss (linear scale — late training)')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('convergence.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 9 — Price surface: V vs S (1D slice)
model.eval()
with torch.no_grad():
    S_grid = torch.linspace(0.5, 2.0, 200).to(device)
    N = len(S_grid)

    def price_slice(tau_val, v_val, r_val):
        tau_t = torch.full((N,), tau_val).to(device)
        v_t   = torch.full((N,), v_val).to(device)
        r_t   = torch.full((N,), r_val).to(device)
        return model(tau_t, S_grid, v_t, r_t).cpu().numpy()

    S_np = S_grid.cpu().numpy()
    intrinsic = np.maximum(S_np - params['K'], 0)

    V_1y  = price_slice(1.0, params['theta'], params['b'])  # 1yr, ATM vol/rate
    V_6m  = price_slice(0.5, params['theta'], params['b'])  # 6m
    V_1m  = price_slice(1/12, params['theta'], params['b']) # 1m
    V_hi_v = price_slice(1.0, 0.12, params['b'])            # high variance
    V_lo_v = price_slice(1.0, 0.01, params['b'])            # low variance

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Term structure
axes[0].plot(S_np, V_1y,  color='#7469d4', lw=2.5, label='τ = 1y')
axes[0].plot(S_np, V_6m,  color='#1D9E75', lw=2.0, label='τ = 6m')
axes[0].plot(S_np, V_1m,  color='#D85A30', lw=2.0, label='τ = 1m')
axes[0].plot(S_np, intrinsic, 'k--', lw=1, alpha=0.5, label='intrinsic value')
axes[0].axvline(params['K'], color='gray', ls=':', lw=1)
axes[0].set_xlabel('Stock price S'); axes[0].set_ylabel('Option value V')
axes[0].set_title('Term structure  (v=θ, r=b)')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Vol sensitivity
axes[1].plot(S_np, V_hi_v, color='#D85A30', lw=2.5, label='v = 0.12 (σ≈35%)')
axes[1].plot(S_np, V_1y,   color='#7469d4', lw=2.5, label='v = θ = 0.04 (σ=20%)')
axes[1].plot(S_np, V_lo_v, color='#1D9E75', lw=2.5, label='v = 0.01 (σ=10%)')
axes[1].plot(S_np, intrinsic, 'k--', lw=1, alpha=0.5, label='intrinsic')
axes[1].axvline(params['K'], color='gray', ls=':', lw=1)
axes[1].set_xlabel('Stock price S'); axes[1].set_ylabel('Option value V')
axes[1].set_title('Vol sensitivity  (τ=1y, r=b)')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('price_slices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 10 — 2D price surface (S vs v heatmap)
model.eval()
with torch.no_grad():
    n = 60
    S_ax = torch.linspace(0.5, 2.0, n)
    v_ax = torch.linspace(0.005, 0.20, n)
    SS, VV = torch.meshgrid(S_ax, v_ax, indexing='ij')  # (n, n)
    SS_flat = SS.reshape(-1).to(device)
    VV_flat = VV.reshape(-1).to(device)
    tau_flat = torch.full_like(SS_flat, 1.0)
    r_flat   = torch.full_like(SS_flat, params['b'])
    price_flat = model(tau_flat, SS_flat, VV_flat, r_flat).cpu().numpy()
    price_grid = price_flat.reshape(n, n)

fig, ax = plt.subplots(figsize=(8, 5))
cf = ax.contourf(S_ax.numpy(), v_ax.numpy(), price_grid.T, levels=30, cmap='RdPu')
plt.colorbar(cf, ax=ax, label='Option price V')
ax.set_xlabel('Stock price S')
ax.set_ylabel('Variance v')
ax.set_title('Price surface  (τ=1y, r=b)  — S × v slice')
ax.axvline(params['K'], color='white', ls='--', lw=1.2, alpha=0.8, label='K=1')
ax.legend()
plt.tight_layout()
plt.savefig('price_surface.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 11 — Greeks via autograd (Delta, Gamma, Vega, Rho)
model.eval()

S_grid_g = torch.linspace(0.6, 1.6, 100).to(device)
S_grid_g.requires_grad_(True)
tau_g = torch.full((100,), 1.0, device=device)
v_g   = torch.full((100,), params['theta'], device=device, requires_grad=True)
r_g   = torch.full((100,), params['b'], device=device, requires_grad=True)

V_g = model(tau_g, S_grid_g, v_g, r_g)

def autograd(y, x):
    return torch.autograd.grad(y, x,
        grad_outputs=torch.ones_like(y),
        create_graph=True, retain_graph=True)[0]

delta = autograd(V_g, S_grid_g)
gamma = autograd(delta, S_grid_g)
vega  = autograd(V_g, v_g)     # ∂V/∂v  (then ∂V/∂σ = 2σ · vega_v)
rho   = autograd(V_g, r_g)     # ∂V/∂r

S_np  = S_grid_g.detach().cpu().numpy()
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, y, name, color in zip(
    axes.flat,
    [delta, gamma, vega, rho],
    ['Delta  ∂V/∂S', 'Gamma  ∂²V/∂S²', 'Vega  ∂V/∂v', 'Rho  ∂V/∂r'],
    ['#7469d4', '#D85A30', '#1D9E75', '#BA7517']
):
    ax.plot(S_np, y.detach().cpu().numpy(), color=color, lw=2)
    ax.axvline(params['K'], color='gray', ls='--', lw=1)
    ax.set_xlabel('S'); ax.set_ylabel(name); ax.set_title(name)
    ax.grid(alpha=0.3)

plt.suptitle('Greeks via autograd  (τ=1y, v=θ, r=b)', y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig('greeks.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 12 — Benchmark: compare vs Black-Scholes (flat vol approximation)
# For a rough sanity check — true benchmark is semi-analytic Heston CF
from scipy.stats import norm
from scipy.optimize import brentq

def bs_call(S, K, T, r, sigma):
    """Black-Scholes call price."""
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)

def bs_iv(price, S, K, T, r):
    """Implied vol from market price via Brent."""
    try:
        return brentq(lambda sig: bs_call(S, K, T, r, sig) - price, 1e-6, 5.0)
    except:
        return np.nan

# PINN prices at τ=1y, v=θ, r=b across strikes
K_grid = np.array([0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3])
model.eval()
pinn_prices, ivs = [], []

with torch.no_grad():
    for K_val in K_grid:
        # Use S=1 (normalised), vary K
        tau_t = torch.tensor([1.0]).to(device)
        S_t   = torch.tensor([1.0]).to(device)
        v_t   = torch.tensor([params['theta']]).to(device)
        r_t   = torch.tensor([params['b']]).to(device)
        p_val = model(tau_t, S_t / K_val, v_t, r_t).item() / K_val
        pinn_prices.append(p_val)
        ivs.append(bs_iv(p_val, 1.0, K_val, 1.0, params['b']))

# BS flat vol reference (sigma = sqrt(theta))
bs_flat = np.sqrt(params['theta'])
bs_prices = [bs_call(1.0, K, 1.0, params['b'], bs_flat) for K in K_grid]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(K_grid, pinn_prices, 'o-', color='#7469d4', lw=2, label='PINN (Heston-HW)')
axes[0].plot(K_grid, bs_prices, 's--', color='gray', lw=1.5, label=f'BS flat σ={bs_flat:.0%}')
axes[0].set_xlabel('Strike K'); axes[0].set_ylabel('Call price (S=1)')
axes[0].set_title('PINN vs Black-Scholes  (S=1, τ=1y)')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(K_grid, np.array(ivs)*100, 'o-', color='#D85A30', lw=2, label='PINN implied vol')
axes[1].axhline(bs_flat*100, color='gray', ls='--', lw=1.5, label=f'BS flat {bs_flat:.0%}')
axes[1].set_xlabel('Strike K'); axes[1].set_ylabel('Implied vol (%)')
axes[1].set_title('Implied volatility smile from PINN')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('smile.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nStrike | PINN price | BS price | Implied vol')
print('-'*50)
for K, pp, bp, iv in zip(K_grid, pinn_prices, bs_prices, ivs):
    print(f'  {K:.2f}  |  {pp:.5f}   | {bp:.5f}  | {iv*100:.2f}%')

In [ ]:
# Cell 13 — Save model
torch.save({
    'model_state': model.state_dict(),
    'params': params,
    'history': history,
}, 'heston_hw_pinn.pt')
print('Model saved to heston_hw_pinn.pt')

# To reload later:
# checkpoint = torch.load('heston_hw_pinn.pt')
# model.load_state_dict(checkpoint['model_state'])
# model.eval()

## Things for experimentation and work

| Experiment | How |
|---|---|
| **Frequency bandwidth** | Change `sigma` in `HestonHWPINN(sigma=...)` — try 0.5, 1.5, 3.0 |
| **Network depth** | Change `depth=4` → 6 for more expressive capacity |
| **LBFGS fine-tune** | After Adam, run 1000 steps of `torch.optim.LBFGS` for tighter residuals |
| **American option** | Add `V ≥ max(S-K,0)` constraint via penalty or projected gradient |
| **True benchmark** | Compare against semi-analytic Heston CF price (via `py_vollib`) |
| **Higher resolution** | Increase `N_INT` to 8192, `N_EPOCHS` to 30000 |
| **Exotic payoff** | Change terminal BC to barrier, Asian, or digital payoff |